# Jobsheet 5

## Setup Environment

In [ ]:
%pip install -q transformers

In [ ]:
import os, time, random, json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision

def set_seed(seed=42):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CEPAT = DEVICE == "cpu"     # tanpa GPU: jalankan versi diperkecil
CEPAT = DEVICE == "cpu"   # tanpa GPU: jalankan versi diperkecil
print("PyTorch", torch.__version__, "| torchvision", torchvision.__version__)
print("Perangkat:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU saja")

CFG = dict(
    IMG     = 128 if CEPAT else 256,    #ukuran sisi citar (pixel)
    N_TRAIN = 320 if CEPAT else 2000,   # jumlah citra latih (dari daftar traincal resmi)
    N_TEST  = 100 if CEPAT else 500,    # jumlah citra uji (dari daftar testcal resmi)
    BATCH   = 16,
    EP_UNET = 6 if CEPAT else 15,
    EP_DLV3 = 2 if CEPAT else 6,
    EP_SEGF = 3 if CEPAT else 6,
    BASE_CH = 16 if CEPAT else 32       # lebar channel unet
)

print("Mode:", "CEPAT (CPU)" if CEPAT else "PENUH (GPU)", CFG)
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)
WAKTU = {}   # catatan waktu tiap bagian

## Praktikum D1: Mengenal Data Segmentasi Oxford-IIIT Pet

 nilai 1 untuk hewan, 2 untuk latar, dan 3 untuk batas yang ragu-ragu.Kita ubah trimap menjadi masker biner (hewan = 1, latar = 0) dan masker valid yang bernilai 0 pada piksel batas.